# Homework 6 — CNN Applications, Transfer Learning, and Pretrained Models (PyTorch)
**CSCI1409 Deep Learning | Aidyn Abirov | Week 6**

Fill in every `# TODO`. Do **not** change the `assert` lines: they tell you when a cell is correct. Write your explanations in the **Your answer** cells.

- **Tasks 1–2: no AI.** Task 3: AI allowed for explanations and debugging only (record which tool and what for).
- Save as `Surname_Firstname_Homework06.ipynb`, restart the kernel, **Run All**, and submit via Teams.
- Needs internet the first time: the ants/bees images (about 45 MB) and the ResNet18 weights (about 45 MB) are downloaded once. A GPU is not needed; Google Colab works too.

In [ ]:
import os, time, zipfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)

# 1. Download the ants vs. bees dataset (once)
if not os.path.isdir("hymenoptera_data"):
    urllib.request.urlretrieve("https://download.pytorch.org/tutorial/hymenoptera_data.zip", "hymenoptera_data.zip")
    with zipfile.ZipFile("hymenoptera_data.zip") as z:
        z.extractall(".")

# 2. The same preprocessing that was used when ResNet18 was trained on ImageNet
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
preprocess = T.Compose([
    T.Resize(256),          # shorter side -> 256 pixels
    T.CenterCrop(224),      # cut out the central 224 x 224 square
    T.ToTensor(),           # pixels 0..255 -> 0..1, shape (3, 224, 224)
    T.Normalize(mean, std), # (x - mean) / std for each colour channel
])

train_set = torchvision.datasets.ImageFolder("hymenoptera_data/train", transform=preprocess)
val_set = torchvision.datasets.ImageFolder("hymenoptera_data/val", transform=preprocess)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=32)
print("classes:", train_set.classes, "| train images:", len(train_set), "| val images:", len(val_set))

# 3. Pretrained weights (ImageNet: 1.2 million photos, 1000 classes)
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
categories = weights.meta["categories"]       # the 1000 ImageNet class names

def show(img):
    """Undo the normalisation so a preprocessed image can be displayed."""
    img = img * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    return img.permute(1, 2, 0).clamp(0, 1)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, np.linspace(0, len(train_set) - 1, 8).astype(int)):
    img, label = train_set[i]
    ax.imshow(show(img)); ax.set_title(train_set.classes[label]); ax.axis("off")
plt.show()

## Task 1 — Use a pretrained model as it is (no AI, about 25 minutes)

**1a. Normalisation by hand.** One pixel has colour $(R, G, B) = (255, 128, 0)$. `ToTensor` divides by 255; then `Normalize` computes $(x - \text{mean}) / \text{std}$ for each channel, with mean $= (0.485, 0.456, 0.406)$ and std $= (0.229, 0.224, 0.225)$. Compute the three normalised values on paper (3 decimals), type them in, and run the check.

In [ ]:
# TODO: your three hand-computed values
hand_normalized = torch.tensor([0.0, 0.0, 0.0])

pixel = torch.tensor([255., 128., 0.]).view(3, 1, 1) / 255     # one pixel as a (3, 1, 1) image
torch_normalized = T.Normalize(mean, std)(pixel).flatten()
print(torch_normalized)
assert torch.allclose(hand_normalized, torch_normalized, atol=1e-2)

**1b. Ask ResNet18 what it sees.** Load the pretrained model and classify four validation images. Complete the two TODO lines: turn the logits into probabilities with softmax, then take the 5 largest with `torch.topk`.

In [ ]:
model = torchvision.models.resnet18(weights=weights)
model.eval()                                   # evaluation mode (important for BatchNorm layers)

idx = [0, 1, len(val_set) - 2, len(val_set) - 1]   # two ants and two bees
batch = torch.stack([val_set[i][0] for i in idx])  # shape (4, 3, 224, 224)
with torch.no_grad():
    logits = model(batch)                      # shape (4, 1000): one score per ImageNet class

probs = None                                   # TODO: softmax over the 1000 classes (which dim?)
top_probs, top_idx = None, None                # TODO: the 5 largest probabilities and their class indices
assert torch.allclose(probs.sum(dim=1), torch.ones(4)) and top_idx.shape == (4, 5)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for k, (ax, i) in enumerate(zip(axes, idx)):
    ax.imshow(show(val_set[i][0])); ax.axis("off")
    ax.set_title(f"true: {val_set.classes[val_set[i][1]]}\n{categories[top_idx[k, 0]]} ({top_probs[k, 0]:.0%})")
    print(f"image {k}:", ", ".join(f"{categories[c]} {p:.1%}" for p, c in zip(top_probs[k], top_idx[k])))
plt.show()

**1c. What happens without normalisation?** Run this cell: it classifies the same four images again but skips `Normalize`. Compare the top-1 predictions with 1b.

In [ ]:
no_norm = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
raw_batch = torch.stack([no_norm(val_set.loader(val_set.samples[i][0])) for i in idx])
with torch.no_grad():
    raw_probs = F.softmax(model(raw_batch), dim=1)
for k in range(4):
    p, c = raw_probs[k].max(dim=0)
    print(f"image {k}: with Normalize -> {categories[logits[k].argmax()]:20s} without -> {categories[c]} ({p:.1%})")

**1d. Look inside.** Print the model, find its last layer, and count parameters. First compute the number of parameters of the last layer **by hand** ($n_{in} \times n_{out} + n_{out}$).

In [ ]:
print(model)                                   # scroll to the end: the last layer is called fc

n_params = 0                                   # TODO: total number of parameters in the model (one line, as in Homework 5)
fc_params_by_hand = 0                          # TODO: parameters of model.fc, computed by hand

print(f"last layer: Linear({model.fc.in_features} -> {model.fc.out_features})")
print(f"ResNet18: {n_params:,} parameters (your Homework 5 CNN had 9,098)")
assert n_params == 11_689_512
assert fc_params_by_hand == sum(p.numel() for p in model.fc.parameters())

**1e. The first-layer filters.** In Homework 5 your CNN learned 8 grayscale 3×3 filters. ResNet18's first layer has 64 colour filters of size 7×7. Run the cell and compare.

In [ ]:
filters = model.conv1.weight.detach()                     # shape (64, 3, 7, 7)
print("conv1 weight shape:", tuple(filters.shape))
fig, axes = plt.subplots(4, 16, figsize=(14, 3.8))
for ax, f in zip(axes.ravel(), filters):
    f = (f - f.min()) / (f.max() - f.min())                # rescale to 0..1 for display
    ax.imshow(f.permute(1, 2, 0)); ax.axis("off")
fig.suptitle("ResNet18 conv1 filters (learned on ImageNet)"); plt.show()

**Questions 1.**
1. Are the top predictions in 1b sensible? Look at the 1000 class names (`categories`): does ImageNet already contain ants and bees?
2. Why must we normalise our images with the **ImageNet** mean and std? What did you see in 1c when normalisation was skipped?
3. Compare the filters in 1e with the ones you learned in Homework 5. What looks similar, and what can these filters detect that yours could not?

**Your answer:**

## Task 2 — Pretrained features and CNN applications (no AI, about 25 minutes)

**2a. Cosine similarity by hand.** $\cos(u, v) = \dfrac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert}$. For $a = (1, 0, 1, 0)$, $b = (1, 1, 0, 0)$, $c = (2, 0, 2, 0)$, $d = (0, 1, 0, 1)$, compute $\cos(a, b)$, $\cos(a, c)$ and $\cos(a, d)$ on paper. Then complete `cosine_similarity` (do not call `F.cosine_similarity` inside it).

In [ ]:
a = torch.tensor([1., 0., 1., 0.]); b = torch.tensor([1., 1., 0., 0.])
c = torch.tensor([2., 0., 2., 0.]); d = torch.tensor([0., 1., 0., 1.])

# TODO: your hand results for cos(a, b), cos(a, c), cos(a, d)
hand_cos = torch.tensor([0.0, 0.0, 0.0])

def cosine_similarity(u, v):
    return 0.0   # TODO: dot product divided by the product of the lengths (hint: torch.dot, torch.norm)

mine = torch.tensor([cosine_similarity(a, v) for v in (b, c, d)])
print(mine)
assert torch.allclose(mine, hand_cos, atol=1e-3)
assert torch.allclose(mine, torch.stack([F.cosine_similarity(a, v, dim=0) for v in (b, c, d)]), atol=1e-6)

**2b. ResNet18 as a feature extractor.** If we remove the last layer, ResNet18 turns every image into a vector of 512 numbers (a *feature vector*) instead of 1000 class scores. Replace `backbone.fc` with `nn.Identity()`, a layer that returns its input unchanged. The cell then computes the feature vectors of all validation images (about 20 seconds).

In [ ]:
backbone = torchvision.models.resnet18(weights=weights)
# TODO: replace the last layer with nn.Identity()

backbone.eval()
features, labels = [], []
with torch.no_grad():
    for images, y in val_loader:
        features.append(backbone(images)); labels.append(y)
features, labels = torch.cat(features), torch.cat(labels)
print("features:", tuple(features.shape))
assert features.shape == (len(val_set), 512)

**2c. Are ants closer to ants than to bees?** Normalise every feature vector to length 1 (`F.normalize(features, dim=1)`); then one matrix product gives the cosine similarity of **every pair** of images. Complete the TODO and compare the three averages.

In [ ]:
unit = None                                    # TODO: feature vectors scaled to length 1
sims = unit @ unit.T                           # sims[i, j] = cosine similarity of images i and j
assert torch.allclose(torch.diagonal(sims), torch.ones(len(val_set)), atol=1e-4)

ant, bee = labels == 0, labels == 1
off_diag = ~torch.eye(len(val_set), dtype=torch.bool)  # ignore each image's similarity with itself
print(f"ant-ant: {sims[ant][:, ant][off_diag[ant][:, ant]].mean():.3f}")
print(f"bee-bee: {sims[bee][:, bee][off_diag[bee][:, bee]].mean():.3f}")
print(f"ant-bee: {sims[ant][:, bee].mean():.3f}")

**2d. Application: image search.** Find the 5 images most similar to a query image. Nobody trained anything for this — it uses only the pretrained features. Complete the TODO, then try a few other `query` values.

In [ ]:
query = 3
scores = sims[query].clone()
scores[query] = -1                             # do not return the query itself
nearest = None                                 # TODO: indices of the 5 largest scores (torch.topk(...).indices)
assert len(nearest) == 5 and query not in nearest

fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
axes[0].imshow(show(val_set[query][0])); axes[0].set_title(f"QUERY ({val_set.classes[labels[query]]})")
for ax, j in zip(axes[1:], nearest):
    ax.imshow(show(val_set[j][0])); ax.set_title(f"{val_set.classes[labels[j]]}, sim {sims[query, j]:.2f}")
for ax in axes: ax.axis("off")
plt.show()

# Use the nearest neighbour as a classifier: predict the class of the single most similar other image
nn_pred = labels[(sims - 2 * torch.eye(len(val_set))).argmax(dim=1)]
print(f"nearest-neighbour accuracy (no training at all): {(nn_pred == labels).float().mean():.1%}")

**2e. CNN applications.** CNNs solve three main kinds of image tasks:
- **classification:** one label for the whole image ("this is a bee");
- **detection:** a box and a label for every object ("bee at (40, 60, 120, 150)");
- **segmentation:** a label for every pixel ("these pixels are bee, the rest is background").

For each application below, write the task type and what the model outputs.

| application | task type | output |
|---|---|---|
| unlocking a phone with your face | | |
| counting cars in a parking-lot camera | | |
| outlining a tumour in a brain scan | | |
| tagging a holiday photo as "beach" or "mountain" | | |
| finding pedestrians for a self-driving car | | |
| blurring the background in a video call | | |

**Your answer:**

**Questions 2.**
1. In 2a, why is $\cos(a, c) = 1$ even though $c$ is twice as long as $a$? Why is that useful when comparing images?
2. In 2c, which average is the smallest? What does that tell you about the 512 features?
3. ResNet18 was never trained on this particular dataset. Why can its features still find similar ants and bees?

**Your answer:**

## Task 3 — Transfer learning vs. training from scratch (AI allowed, about 45 minutes)
AI may be used to explain concepts or help debug this task only. Record which tool you used and for what, and be ready to explain every line.

We train three models on the 244 training images and compare them on the validation set. `accuracy` and `train` are the same as in Homework 5 and are given. The only new thing in `train` is that it updates **only the parameters with `requires_grad=True`**.

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, y in loader:
            correct += (model(images).argmax(dim=1) == y).sum().item()
    return correct / len(loader.dataset)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train(model, epochs, lr):
    trainable = [p for p in model.parameters() if p.requires_grad]   # frozen parameters are left out
    optimizer = torch.optim.SGD(trainable, lr=lr)
    history = []
    for epoch in range(epochs):
        model.train()
        for images, y in train_loader:
            loss = F.cross_entropy(model(images), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        history.append(accuracy(model, val_loader))
        print(f"  epoch {epoch + 1}: val accuracy {history[-1]:.3f}")
    return history

results = {}   # name -> (trainable parameters, seconds, accuracy history)

**3a. Run 1: a small CNN from scratch.** The Homework 5 CNN, changed for colour (3 input channels) and 224×224 images. Compute the trainable parameters **by hand** first (after two 2×2 poolings, 224 → 56).

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 56 * 56, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        return self.fc(x.flatten(1))

scratch_params_by_hand = 0                     # TODO: conv1 + conv2 + fc
torch.manual_seed(0)
scratch = SmallCNN()
assert scratch_params_by_hand == count_trainable(scratch)

start = time.time(); history = train(scratch, epochs=10, lr=0.01)
results["from scratch"] = (count_trainable(scratch), time.time() - start, history)

**3b. Run 2: feature extraction.** Take pretrained ResNet18, **freeze** all its parameters (`requires_grad = False`), and replace `fc` with a new `nn.Linear(512, 2)`. Only the new layer will be trained.

In [ ]:
def make_feature_extractor():
    model = torchvision.models.resnet18(weights=weights)
    # TODO 1: freeze every parameter of the model (loop over model.parameters())
    # TODO 2: replace model.fc with a new Linear layer: 512 inputs, 2 outputs (ants, bees)
    return model

torch.manual_seed(0)
frozen = make_feature_extractor()
assert count_trainable(frozen) == 1026, "only the new 512 -> 2 layer should be trainable"

start = time.time(); history = train(frozen, epochs=5, lr=0.01)
results["feature extraction"] = (count_trainable(frozen), time.time() - start, history)

**3c. Run 3: fine-tuning.** Start like 3b, then **unfreeze** the last block of ResNet18 (`model.layer4`) so it can adapt to ants and bees too. Use a smaller learning rate so the pretrained weights change only a little.

In [ ]:
def make_finetune():
    model = make_feature_extractor()
    # TODO: set requires_grad = True for every parameter in model.layer4
    return model

torch.manual_seed(0)
finetuned = make_finetune()
layer4_params = sum(p.numel() for p in finetuned.layer4.parameters())
assert count_trainable(finetuned) == layer4_params + 1026

start = time.time(); history = train(finetuned, epochs=5, lr=0.005)
results["fine-tuning (layer4 + fc)"] = (count_trainable(finetuned), time.time() - start, history)

**3d. Compare.** The table and plot are made for you. The last cell shows the validation images the fine-tuned model got wrong.

In [ ]:
print(f"{'run':28s} {'trainable params':>17s} {'time (s)':>9s} {'final val acc':>14s}")
for name, (n, sec, hist) in results.items():
    print(f"{name:28s} {n:17,d} {sec:9.0f} {hist[-1]:14.3f}")

plt.figure(figsize=(7, 4))
for name, (_, _, hist) in results.items():
    plt.plot(range(1, len(hist) + 1), hist, "o-", label=name)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.ylim(0.4, 1.0); plt.legend(); plt.show()

assert results["feature extraction"][2][-1] > 0.85, "feature extraction should beat 85%; check your freezing and new fc"
assert results["feature extraction"][2][-1] > results["from scratch"][2][-1]

In [ ]:
finetuned.eval(); wrong = []
with torch.no_grad():
    for i in range(len(val_set)):
        img, y = val_set[i]
        pred = finetuned(img[None]).argmax(dim=1).item()
        if pred != y: wrong.append((img, pred, y))
print(f"{len(wrong)} of {len(val_set)} validation images misclassified")

if wrong:
    fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(14, 2.6), squeeze=False)
    for ax, (img, pred, y) in zip(axes[0], wrong):
        ax.imshow(show(img)); ax.set_title(f"pred {val_set.classes[pred]}\ntrue {val_set.classes[y]}", fontsize=9); ax.axis("off")
    plt.show()

**Questions 3.**
1. Rank the three runs by validation accuracy. Why does the CNN from scratch do badly with only 244 training images, even though it trains about 100 times more parameters than feature extraction?
2. Feature extraction trains only 1,026 parameters. Where does the rest of the "knowledge" come from?
3. You get a new task: classify 300 chest X-rays as healthy or sick. Would you freeze the backbone or fine-tune more layers? What about a task with 100,000 images? Explain.
4. What does `requires_grad = False` do? Why is the new `fc` layer trainable even though you never set `requires_grad = True` for it?

**Your answer:**

**Optional (not required).** Data augmentation creates new training images by randomly changing existing ones. Retrain the from-scratch CNN with this training transform and compare its accuracy with 3a:
```python
augment = T.Compose([T.RandomResizedCrop(224), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
```
(Create a new `train_set`/`train_loader` with it; keep `val_set` unchanged.)

In [ ]:
# Optional: your code here

## Submission checklist
- [ ] Every `# TODO` filled, every `assert` passes, **Kernel → Restart & Run All** works
- [ ] All **Your answer** cells written (1, 2, 2e table, 3)
- [ ] Hand calculations for 1a, 1d, 2a, and 3a done on paper first
- [ ] Tasks 1–2 done without AI; references listed
- [ ] Task 3 AI use recorded (tool + what for), or "no AI used"

**References / AI use:**